# មេរៀន 13 - ចងចាំរបស់ភ្នាក់ងារ


## ការរៀបចំ

កំណត់សម្គាល់នេះ បង្ហាញពីរបៀបក្នុងការបង្កើតភ្នាក់ងារកក់ដំណើរកំសាន្ត ជាមួយ **ចងចាំយូរអង្វែង** ដោយប្រើ **Microsoft Agent Framework** (MAF).

អ្នកនឹងរៀនពីរបៀបដែលប្រភេទចងចាំផ្សេងៗរបស់ភ្នាក់ងារ — ចងចាំកំពុងដំណើរការ, ចងចាំរយៈពេលខ្លី, និង ចងចាំរយៈពេលវែង — មានឥទ្ធិពលយ៉ាងដូចម្តេចលើរបៀបដែលភ្នាក់ងារកាន់កាប់ និងប្រើព័ត៌មាននៅក្នុងការសន្ទនានានា។

**លក្ខខណ្ឌតម្រូវ៖**
- គម្រោង Azure AI Foundry ដែលមានម៉ូឌែលជជែកបានដាក់ចេញ (ឧទាហរណ៍ `gpt-4o-mini`).
- បានចូលជាមួយ Azure CLI — ដំណើរការ `az login` នៅក្នុងផ្ទាំងបញ្ជារបស់អ្នក។
- `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចផ្តល់សេវារបស់គម្រោង Azure AI Foundry របស់អ្នក។
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះនៃម៉ូឌែលដែលអ្នកបានដាក់ចេញ។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ប្រភេទនៃចងចាំភ្នាក់ងារ

ភ្នាក់ងារ AI អាចប្រើប្រាស់ចងចាំប្រភេទផ្សេងៗ នីមួយៗមានគោលបំណងខុសគ្នា:

### ចងចាំកំពុង​ធ្វើការ
ខ្សែកិច្ចសន្ទនាเอง — សារ​ដែលបានប្តូរវិញក្នុងសម័យតែមួយ។ ភ្នាក់ងារអាចយោងទៅកាន់សារចាស់ៗក្នុងខ្សែដដែលដើម្បីរក្សាភាពស្របគ្នា។ នៅក្នុង MAF នេះត្រូវបានបង្កើតតាមរយៈ **`agent.create_session()`**, ដែល​នឹង​ត្រឡប់​តម្លៃជា `AgentSession`.

### ចងចាំ​រយៈពេលខ្លី
ព័ត៌មានដែលនៅស្ថិតសម្រាប់រយៈពេលនៃភារកិច្ចឬសម័យ ប៉ុន្តែមិនត្រូវបានផ្ទុកយ៉ាងអចិន្រ្តៃយ៍។ ឧទាហរណ៍ ភ្នាក់ងារ​អាចប្រមូលព័ត៌មាននៅក្នុងការសន្ទនាធ្វើផែនការជាច្រើនជំហាន ហើយប្រើព័ត៍មានទាំងនោះដើម្បីបង្កើតកម្មវិធីដំណើរចុងក្រោយ។

### ចងចាំ​រយៈពេលវែង
ចំណូលចិត្ត និងព័ត៌មានដែលនៅស្ថិត **ឆ្លងកាត់សម័យ**។ អ្នកប្រើដែលត្រឡប់មកម្តងទៀតមិនគួរត្រូវបញ្ចាក់ពីកំណត់អាហារ ឬរបៀបធ្វើដំណើររបស់ពួកគេម្ដងទៀត។ ចងចាំរយៈពេលវែងភាគច្រើនត្រូវបានគាំទ្រដោយឃ្លាំងខាងក្រៅ — មូលដ្ឋានទិន្នន័យ, ឯកសារ, ឬសន្ទស្សន៍វ៉ិចទ័រ — ហើយត្រូវបានបង្ហាញទៅភ្នាក់ងារដោយឧបករណ៍។


## ចងចាំកំពុង​ធ្វើការ ជាមួយសម័យ

ទម្រង់សាមញ្ញបំផុតនៃចងចាំគឺសម័យនៃការសន្ទនា។ នៅពេលអ្នកផ្ញើសម័យដូចគ្នា (បង្កើតតាម `agent.create_session()`) ទៅការហៅ `agent.run()` ជាប់ៗគ្នា អ្នកជំនួយនឹងឃើញប្រវត្តិពេញលេញនៃការសន្ទនានោះ និងអាចរំលឹកព័ត៌មានពីមុនៗបាន។

មកបង្កើតភ្នាក់ងារធ្វើដំណើរមួយ ហើយបង្ហាញពីចងចាំកំពុងធ្វើការរបស់វា។


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ភ្នាក់ងារ​បាន​ចងចាំ​ថវិកា​បាន​យ៉ាង​ត្រឹមត្រូវ ព្រោះ​សារ​ទាំងពីរ​ចែករំលែក​សម័យ​ដូច​គ្នា។ នេះ​គឺជា **ចងចាំ​សកម្ម** — វា​មាន​តែ​នៅ​រហូត​ដល់​អាយុកាល​នៃ​សម័យ​នោះ​តែប៉ុណ្ណោះ។

### តើ​មាន​អ្វី​កើត​ឡើង​នៅ​ពេល​មាន​ខ្សែ​សន្ទនា​ថ្មី?

បើ​ពួក​យើង​បង្កើត​សម័យ **ថ្មី** ភ្នាក់ងារ​នោះ​មិន​មាន​ការ​ចងចាំ​ពី​សន្ទនាកន្លងមក​ទេ៖


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## លំនាំចងចាំរយៈពេលវែង

ដើម្បីចងចាំចំណូលចិត្តរបស់អ្នកប្រើ **ឆ្លងកាត់សម័យ** យើងត្រូវការឃ្លាំងដែលមានភាពរឹងមាំ ហើយរស់នៅក្រៅខ្សែសន្ទនា។ ភ្នាក់ងារ ចូលដំណើរការឃ្លាំងនេះតាមរយៈ **ឧបករណ៍** — អនុគមន៍ដែលវាអាចហៅដើម្បីរក្សាទុក និងយកព័ត៌មាន។

ខាងក្រោម យើងអនុវត្តិហាងចំណូលចិត្តសាមញ្ញមួយក្នុងម៉េមរី (ក្នុងការផលិត អ្នកនឹងគាំទ្រវាដោយមូលដ្ឋានទិន្នន័យ ឬ សន្ទស្សន៍វ៉ិចទ័រ) ហើយបង្ហាញវាជា ឧបករណ៍ដែលភ្នាក់ងារ​អាចប្រើ។

### ស្ថាបត្យកម្ម
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### សេណារីយ៉ូ 1 — អ្នកប្រើប្រាស់លើកដំបូងកក់ដំណើរកម្សាន្តសម្រាប់ខួប

Sarah មកទស្សនាលើកដំបូង។ ភ្នាក់ងារគួរតែរក្សាទុកចំណង់ចំណូលចិត្តរបស់នាងតាមរយៈឧបករណ៍ ហើយប្រើវាដើម្បីផ្ដល់អនុសាសន៍សណ្ឋាគារ។


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### សេណារីយ៉ូ 2 — Sarah ត្រឡប់មកវិញបន្ទាប់ពីប៉ុន្មានសប្តាហ៍

Sarah ចាប់ផ្តើម **សន្ទនាថ្មីទាំងស្រុង** (ដែលរៀបចំដូចជា​សម័យថ្មី)។ អង្គចងចាំការងារកំពុងទទេ ប៉ុន្មែកឃ្លាំងចំណូលចិត្តរយៈពេលវែងនៅតែមានព័ត៌មានរបស់នាង។ ភ្នាក់ងារត្រូវយកវាមកវិញ ហើយប្រើវា ដើម្បីផ្គូផ្គងអនុសាសន៍ឱ្យផ្ទាល់ខ្លួន។


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## សេចក្តីសង្ខេប

នៅមេរៀននេះ អ្នកបានរៀនពីប្រភេទចងចាំ agent ចំនួនបី និងរបៀបអនុវត្តពួកវាជាមួយ Microsoft Agent Framework៖

| ប្រភេទចងចាំ | MAF Mechanism | អាយុកាល |
|---|---|---|
| **ចងចាំបច្ចុប្បន្ន** | `agent.create_session()` | សន្ទនាតែមួយ |
| **រយៈពេលខ្លី** | បរិបទដែលបានប្រមូលក្នុង thread | ភារកិច្ច / សេស្យុង តែមួយ |
| **រយៈពេលយូរ** | ឃ្លាំងខាងក្រៅដែលចូលប្រើតាមរយៈ `@tool` functions | ឆ្លងកាត់សេស្យុងជាច្រើន |

### ចំណុចសំខាន់
1. **`agent.create_session()`** ផ្តល់នូវចងចាំបច្ចុប្បន្ន — ភ្នាក់ងារមើលឃើញប្រវត្តិការសន្ទនាទាំងមូលនៅក្នុងសេស្យុងមួយ។
2. **សេស្យុងថ្មីបាត់បង់បរិបទ** — បើគ្មានចងចាំរយៈពេលយូរ ភ្នាក់ងារមិនអាចចងចាំការសន្ទនាមុនៗបាន។
3. **`@tool` functions** ជាស្ពានដើម្បីភ្ជាប់ — ពួកវាអនុញ្ញាតឲ្យភ្នាក់ងាររក្សាទុក និងយកព័ត៌មានពីឃ្លាំងដែលអាចរក្សាទុកបានយូរ។
4. **ការផ្ទាល់ខ្លួនកាន់តែប្រសើរឡើងជាមួយពេលវេលา** — semakin ចំនួនចំណង់ចំណូលចិត្តដែលបានរក្សាទុក ភ្នាក់ងារនឹងផ្តល់អនុសាសន៍បានល្អប្រសើរជាងមុន។

### កម្មវិធីក្នុងពិភពពិត
- **សេវាកម្មអតិថិជន**: ចងចាំប្រវត្តិ និងចំណង់ចំណូលចិត្តរបស់អតិថិជន
- **ជំនួយការផ្ទាល់ខ្លួន**: ថែរក្សាបរិបទឆ្លងកាត់ថ្ងៃឬសប្តាហ៍
- **ការថែទាំសុខាភិបាល**: តាមដានព័ត៌មានអ្នកជម្ងឺ និងចំណង់ចំណូលចិត្ត
- **ពាណិជ្ជកម្មអេឡិចត្រូនិក (E-commerce)**: ការទិញផ្ទាល់ខ្លួនផ្អែកលើប្រវត្តិ

### ជំហានបន្ទាប់
- ជំនួស in-memory dict ជាមួយមូលដ្ឋានទិន្នន័យ ឬ vector store (ឧ. Azure AI Search)
- បន្ថែមការផុតកំណត់ចងចាំសម្រាប់ព័ត៌មានដែលពាក់ព័ន្ធនឹងពេលវេលា
- សាងសង់ប្រព័ន្ធ multi-agent ជាមួយចងចាំរួម
- ស្វែងរក Cognee notebook សម្រាប់ចងចាំដែលផ្អែកលើ knowledge-graph


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំបំពេញភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយដំណើរការ​ស្វ័យប្រវត្តិ​អាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាមាតុភូមិគួរត្រូវបានចាត់ទុកជាប្រភពផ្លូវការដែលមានអាទិភាព។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតមានពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
